In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [2]:
import openpyxl
datalink = '240823_Gemini작업완료.xlsx'

indicators = ['connectivity','measurability','directivity']
data = pd.read_excel(datalink)

In [11]:
def loss(df, answer, predict):
    answer_tensor = torch.empty(0)
    predict_tensor = torch.empty(0)

    for ans, pre in zip(answer, predict):
        df_clean = df.dropna(subset=[ans, pre]) # 해당 열에 정보가 없는 부분 제거

        # df_clean이 비었다면 다음으로 넘어감
        if df_clean.empty:
            continue

        answer_temp = torch.tensor(df_clean[[ans]].values, dtype=int).reshape(-1) # answer를 1차원 tensor로 변환
        predict_temp = torch.tensor(df_clean[[pre]].values, dtype=int).reshape(-1) # predict를 1차원 tensor로 변환
        
        answer_tensor = torch.cat((answer_tensor, answer_temp), dim=0) # 1차원 answer를 합침
        predict_tensor = torch.cat((predict_tensor, predict_temp), dim=0) # 1차원 predict를 합침

    mae_loss = nn.L1Loss() # L1 Loss
    mse_loss = nn.MSELoss() # Mean Squared Error Loss(L2 Loss)

    loss_mae = mae_loss(predict_tensor, answer_tensor) #평가 결과의 l1 loss
    loss_mse = mse_loss(predict_tensor, answer_tensor) #평가 결과의 l2 loss

    print(f'MAE(L1) Loss Objective[{", ".join(predict)}] : {loss_mae.item()}')
    print(f'MSE(L2) Loss Objective[{", ".join(predict)}] : {loss_mse.item()}')

In [ ]:
for a in indicators:
    b = 'predict_' + a
    loss(data, [a], [b]) # loss 함수 호출

In [ ]:
for a in indicators:
  # CSV 파일에서 데이터 읽기
  predict_indicators = 'predict_'+a
  b = predict_indicators
  
  temp_data_clean = data.dropna(subset=[a])
  data_clean = temp_data_clean.dropna(subset=[b])
  print(f"예측 결과 없음 : {len(temp_data_clean)-len(data_clean)}")

  temp = [[0 for _ in range(5)] for _ in range(5)]

  num = 0
  acc0 = 0
  acc1 = 0
  acc2 = 0
  for idx, row in data_clean.iterrows():
    temp[int(row[a])-1][int(row[b])-1] += 1
    num += 1
    
    if (int(row[a])- int(row[b])==0):
      acc0 += 1
    elif (int(row[a])- int(row[b])==1 or int(row[a])- int(row[b])==-1):
      acc1 += 1
    else:
      acc2 += 1

  print(f"acc0 : {acc0/num:.2f}")
  print(f"acc1 : {acc1/num:.2f}")
  print(f"acc2 : {acc2/num:.2f}")
 
  print(f"<{predict_indicators}>")
  for i in range(5):
    for j in range(5):
        print(temp[i][j], end=' ')
    print()
  print() 
  
  # 실제 점수와 예측 점수 추출
  actual_scores = data_clean[a].astype(int)
  predicted_scores = data_clean[b].astype(int)

  # 다중 클래스 분류의 정밀도 계산
  precision = precision_score(actual_scores, predicted_scores, average='weighted')

  print(f'{a}의 Precision: {precision:.2f}')

In [ ]:
for a in indicators:  
  predict_indicators = 'predict_'+a
  b = predict_indicators
  
  data_clean = data.dropna(subset=[a,b])
  
  # 실제 점수와 예측 점수 추출
  actual_scores = data_clean[a].astype(int)
  predicted_scores = data_clean[b].astype(int)

  # 다중 클래스 분류의 recall 계산
  recall = recall_score(actual_scores, predicted_scores, average='weighted')

  print(f'{a}의 recall: {recall:.2f}')

In [ ]:
for a in indicators:
  predict_indicators = 'predict_'+a
  b = predict_indicators
  
  data_clean = data.dropna(subset=[a,b])


  # 실제 점수와 예측 점수 추출
  actual_scores = data_clean[a].astype(int)
  predicted_scores = data_clean[b].astype(int)

  # 다중 클래스 분류의 accuracy 계산
  accuracy = accuracy_score(actual_scores, predicted_scores)

  print(f'{a}의 accuracy: {accuracy:.2f}')

In [ ]:
for a in indicators:
  predict_indicators = 'predict_' + a
  b = predict_indicators

  # 빈 값을 가진 행 제거
  data_clean = data.dropna(subset=[a, b])

  # 실제 점수와 예측 점수 추출
  actual_scores = data_clean[a].astype(int)
  predicted_scores = data_clean[b].astype(int)

  # 다중 클래스 분류의 F1 스코어 계산
  f1 = f1_score(actual_scores, predicted_scores, average='weighted')
  
  print(f'{a}의 F1 Score: {f1:.2f}')